# Economic Correlation Analysis: Interest Rates & Growth

This notebook demonstrates the correlation analysis between interest rates and economic growth indicators.

In [ ]:
# Setup
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.loaders import EconomicDataLoader
from analysis.correlation import CorrelationAnalyzer
from visualization.plots import EconomicPlotter

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Load Economic Data

In [ ]:
# Initialize data loader
loader = EconomicDataLoader(start_date='2000-01-01')

# Load interest rates
print("Loading interest rate data...")
try:
    fed_funds = loader.get_series('fed_funds_rate', 'Federal Funds Rate')
    treasury_10y = loader.get_series('treasury_10y', '10-Year Treasury')
    print(f"✓ Loaded {len(fed_funds)} observations")
except Exception as e:
    print(f"Error: {e}")
    print("Using synthetic data for demonstration...")
    # Create synthetic data
    dates = pd.date_range(start='2000-01-01', end='2024-12-31', freq='M')
    np.random.seed(42)
    trend = np.linspace(6.5, 4.5, len(dates))
    fed_funds = pd.Series(
        trend + np.random.randn(len(dates)) * 0.5 + np.sin(np.arange(len(dates)) * 0.2) * 1.5,
        index=dates,
        name='Federal Funds Rate'
    ).clip(lower=0)
    treasury_10y = fed_funds + 1.5

In [ ]:
# Load economic indicators
print("Loading economic indicators...")
try:
    gdp_growth = loader.get_gdp_growth()
    unemployment = loader.get_unemployment()
    inflation = loader.get_inflation()
    print("✓ Loaded all indicators")
except Exception as e:
    print(f"Error: {e}")
    print("Using synthetic data...")
    gdp_growth = pd.Series(
        3.0 - 0.3 * fed_funds + np.random.randn(len(fed_funds)) * 0.8,
        index=fed_funds.index,
        name='GDP Growth Rate'
    )
    unemployment = pd.Series(
        5.5 + 0.2 * fed_funds + np.random.randn(len(fed_funds)) * 0.5,
        index=fed_funds.index,
        name='Unemployment Rate'
    ).clip(lower=3, upper=10)
    inflation = pd.Series(
        2.5 + 0.15 * fed_funds + np.random.randn(len(fed_funds)) * 0.6,
        index=fed_funds.index,
        name='Inflation Rate'
    )

## 2. Data Overview

In [ ]:
# Combine into DataFrame
data = pd.DataFrame({
    'Fed Funds Rate': fed_funds,
    'GDP Growth': gdp_growth,
    'Unemployment': unemployment,
    'Inflation': inflation
})

# Display first few rows
data.head(10)

In [ ]:
# Summary statistics
analyzer = CorrelationAnalyzer()
summary = analyzer.summary_statistics(data)
summary

## 3. Visualize Time Series

In [ ]:
# Plot all indicators
plotter = EconomicPlotter()
fig = plotter.plot_time_series(data, title="Economic Indicators Over Time", ylabel="Value (%)")
plt.show()

In [ ]:
# Dual axis: Interest Rate vs GDP Growth
fig = plotter.plot_dual_axis(
    fed_funds,
    gdp_growth,
    title="Federal Funds Rate vs GDP Growth"
)
plt.show()

## 4. Correlation Analysis

In [ ]:
# Calculate correlation matrix
corr_matrix, pval_matrix = analyzer.correlation_matrix(data)

print("Correlation Matrix:")
print(corr_matrix)
print("\nP-value Matrix:")
print(pval_matrix)

In [ ]:
# Visualize correlation matrix
fig = plotter.plot_correlation_matrix(
    corr_matrix,
    pval_matrix,
    title="Correlation Matrix: Interest Rates & Economic Indicators"
)
plt.show()

## 5. Interest Rate Impact Analysis

In [ ]:
# Analyze impact on economic indicators
indicators = {
    'GDP Growth': gdp_growth,
    'Unemployment': unemployment,
    'Inflation': inflation
}

results = analyzer.analyze_interest_rate_impact(
    fed_funds,
    indicators,
    lag_analysis=True,
    max_lag=12
)

print("Current Correlations:")
display(results['current_correlations'])

print("\nOptimal Lags:")
display(results['optimal_lags'])

## 6. Scatter Plots

In [ ]:
# Fed Funds vs GDP Growth
fig = plotter.plot_scatter_with_regression(
    fed_funds,
    gdp_growth,
    title="Interest Rate vs GDP Growth"
)
plt.show()

In [ ]:
# Fed Funds vs Unemployment
fig = plotter.plot_scatter_with_regression(
    fed_funds,
    unemployment,
    title="Interest Rate vs Unemployment"
)
plt.show()

## 7. Lagged Correlation Analysis

In [ ]:
# Plot lagged correlations for GDP Growth
lag_df = results['lagged_correlations']['GDP Growth']
fig = plotter.plot_lagged_correlation(lag_df, 'GDP Growth')
plt.show()

In [ ]:
# Plot lagged correlations for all indicators
fig, axes = plt.subplots(len(indicators), 1, figsize=(12, 12))

for i, (name, lag_df) in enumerate(results['lagged_correlations'].items()):
    ax = axes[i]
    colors = ['red' if p < 0.05 else 'gray' for p in lag_df['p_value']]
    ax.bar(lag_df['lag'], lag_df['correlation'], color=colors, alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_title(f'Lagged Correlation: Interest Rate vs {name}')
    ax.set_xlabel('Lag (periods)')
    ax.set_ylabel('Correlation')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Rolling Correlation

In [ ]:
# Calculate rolling correlation
rolling_corr = analyzer.rolling_correlation(fed_funds, gdp_growth, window=12)

# Plot
fig = plotter.plot_rolling_correlation(
    rolling_corr,
    series1_name="Federal Funds Rate",
    series2_name="GDP Growth",
    window=12
)
plt.show()

## 9. Comprehensive Dashboard

In [ ]:
# Create comprehensive dashboard
fig = plotter.create_dashboard(
    fed_funds,
    gdp_growth,
    results
)
plt.show()

## 10. Key Insights

Based on the correlation analysis, we can draw the following conclusions:

1. **Interest Rates vs GDP Growth**: [Interpret the correlation]
2. **Interest Rates vs Unemployment**: [Interpret the correlation]
3. **Interest Rates vs Inflation**: [Interpret the correlation]
4. **Lagged Effects**: [Describe any significant lead-lag relationships]
5. **Time-Varying Relationships**: [Discuss changes in rolling correlation]